## PUCT(Polynomial Upper Confidence Trees) Selection Formula

$$
\boxed{
a^* = \arg\max_a \left[
\underbrace{Q(s,a)}_{\text{Exploitation}}
+
\underbrace{c_{\text{puct}} P(s,a) \frac{\sqrt{N(s)}}{1+N(s,a)}}_{\text{Exploration}}
\right]
}
$$

### Notation

| Symbol | Description |
|---|---|
| $s$ | Current board state |
| $a$ | Candidate action (move) |
| $Q(s,a)$ | Mean action value estimated from search, indicating how promising the move currently appears |
| $P(s,a)$ | Prior probability assigned to the move by the policy network |
| $N(s)$ | Total visit count of the parent node |
| $N(s,a)$ | Number of times this action has been visited during search |
| $c_{\text{puct}}$ | Hyperparameter controlling the strength of exploration |


---

## Why Does a Larger $N(\text{parent})$ Give Underexplored Moves Another Chance?

Let's look at a numerical example.

### 1. The Exploration Term in PUCT

$$
U(s,a) = c_{\text{puct}} P(s,a) \frac{\sqrt{N(\text{parent})}}{1+N(s,a)}
$$

Suppose there are only two possible moves, A and B, with:

$$
\begin{aligned}
c_{\text{puct}} &= 1 \\
P(A) &= P(B) = 0.5 \\
Q(A) &= 0.7 \\
Q(B) &= 0.5
\end{aligned}
$$

Move A currently looks better than move B.

### 2. Scenario 1: After 10 Searches

Suppose A has been visited 9 times and B has been visited once.

| Metric | Move A | Move B |
|---|---:|---:|
| $Q$ | 0.7 | 0.5 |
| $N(a)$ | 9 | 1 |
| $U = \frac{0.5\sqrt{10}}{1+N(a)}$ | 0.158 | 0.791 |
| **$Q+U$** | **0.858** | **1.291** |

**PUCT selects B.**

Although B has a lower estimated value $Q$, it has not been explored sufficiently, so its exploration bonus is much higher.

### 3. Scenario 2: After 100 Searches

Now suppose the parent node has been visited 100 times:

- A has been visited 98 times.
- B has been visited only twice.

For simplicity, assume the $Q$ values remain unchanged.

| Metric | Move A | Move B |
|---|---:|---:|
| $Q$ | 0.7 | 0.5 |
| $N(a)$ | 98 | 2 |
| $U = \frac{0.5\sqrt{100}}{1+N(a)}$ | 0.051 | 1.667 |
| **$Q+U$** | **0.751** | **2.167** |

**B's exploration bonus has actually become even larger!**

Why?

- A has already been explored 98 times. Its large denominator causes its exploration bonus to shrink.
- B has only been explored twice. Its denominator remains small.
- The parent visit count has increased from 10 to 100, causing the numerator $\sqrt{N(\text{parent})}$ to **increase from $\sqrt{10}$ to $10$**.

As a result, B becomes highly attractive again.

**Important:** These two tables are hypothetical snapshots designed to illustrate the formula, not consecutive states from an actual PUCT search trajectory. In a real search, PUCT would not keep ignoring B when its selection score is this high.

### 4. Why Do We Need $\sqrt{N(\text{parent})}$?

Let's go one level deeper.

Suppose we remove the parent visit count from the exploration term:

$$
U(s,a) = \frac{cP(s,a)}{1+N(s,a)}
$$

For a move that has never been explored:

$$
N(s,a)=0
$$

Its exploration bonus would always be:

$$
U=cP
$$

Now imagine that a move has a very low estimated value $Q$.

If its total selection score $Q+U$ remains below that of the currently preferred move, the search might never return to it, regardless of whether we perform 100 searches or 1 million searches.

In other words, **increasing the search budget would not necessarily give neglected moves another chance.**

Now consider what happens when we include $\sqrt{N(\text{parent})}$.

Suppose B remains unexplored:

$$
U(B)=cP(B)\sqrt{N(\text{parent})}
$$

As long as:

$$
P(B)>0
$$

its exploration bonus continues to grow as the parent node receives more visits.

Eventually, that bonus can become large enough to overcome B's lower estimated value, allowing it to be selected for further exploration.

---

## Why Are $P$ and $Q$ Separate? Shouldn't the Move with the Highest Value Also Have the Highest Probability?

$P$ represents the neural network's judgment **before search**, while $Q$ represents the evidence accumulated **during search**.

They are usually correlated, but they do not necessarily agree.

### 1. A Chess Example

Suppose AlphaZero faces three possible moves:

- **A:** Capture the opponent's queen. This looks extremely profitable.
- **B:** Retreat to defend. This looks like an ordinary move.
- **C:** Advance a pawn. Nothing particularly special.

The policy network evaluates the current board position and outputs:

| Move | $P(s,a)$ | Neural Network's Intuition |
|---|---:|---|
| A: Capture the queen | 0.75 | Excellent move! |
| B: Defend | 0.20 | Average |
| C: Advance a pawn | 0.05 | Not worth much attention |

Notice that **no tree search has been performed on these three moves yet.**

Now MCTS begins exploring possible future positions.

It discovers that:

- **A:** Although capturing the queen wins material immediately, it leads to checkmate against AlphaZero three moves later.
- **B:** Although the defensive move looks unremarkable, it prevents defeat and gradually builds an advantage.
- **C:** Advancing the pawn offers no obvious benefit.

Suppose the search produces the following estimates:

| Move | $P(s,a)$ | $Q(s,a)$ |
|---|---:|---:|
| A: Capture the queen | 0.75 | -0.80 |
| B: Defend | 0.20 | +0.70 |
| C: Advance a pawn | 0.05 | -0.10 |

These values are hypothetical. $Q$ uses a value scale from $-1$ (loss) to $+1$ (win).

Now something interesting happens:

**Move A has the highest $P$, but the lowest $Q$!**

Why?

- **$P$ means:** How strongly did the neural network initially prefer this move?
- **$Q$ means:** Based on the evidence accumulated through search, how good does this move currently appear?

Insight:
>MCTS uses **computation to correct** the neural network's **initial intuition.**

Why Does the Policy Network Learn from MCTS Visit Counts Instead of Q-Values?

## 1. The Big Picture: MCTS as a Teacher

**The central idea: MCTS spends substantial computation to discover better decisions, while the Policy Network learns to approximate those decisions without repeating the entire search.**

This creates a powerful feedback loop:

$$
\boxed{
\text{Policy Network}
\xrightarrow{\text{MCTS}}
\text{Improved Policy}
\xrightarrow{\text{Training}}
\text{Better Policy Network}
}
$$

However, this raises an important question:

> If MCTS produces Q-values containing more search information than the original policy probabilities, why does AlphaZero train its Policy Network to predict the MCTS visit-count distribution rather than directly learning from Q-values?

The answer involves the distinction between **value estimation**, **decision-making**, and **computation allocation**.

---

## 2. Example: How MCTS Improves the Original Policy

Suppose AlphaZero is considering three possible moves: A, B, and C.

Before searching, the Policy Network produces:

$$
P(A)=0.70,\quad P(B)=0.20,\quad P(C)=0.10
$$

The network strongly prefers A.

Now suppose MCTS performs 1,000 simulations.

After searching, we obtain the following hypothetical results:

| Move | Original Policy $P$ | Search Q-Value | Visit Count $N$ |
|---|---:|---:|---:|
| A | 70% | +0.2 | 150 |
| B | 20% | +0.8 | 750 |
| C | 10% | +0.5 | 100 |
| **Total** | **100%** | — | **1,000** |

**What happened?**

- Initially, the neural network believed A was the best move.
- MCTS discovered that B had a much better estimated value.
- The search increasingly allocated simulations to B.
- By the end, B had received 75% of all visits.

MCTS has effectively corrected the network's initial intuition.

### Turning Visit Counts into a Training Target

AlphaZero constructs an improved policy by normalizing the visit counts.

For temperature $\tau=1$:

$$
\pi(a|s)=\frac{N(s,a)}{\sum_b N(s,b)}
$$

Therefore:

$$
\pi(A)=0.15,\quad \pi(B)=0.75,\quad \pi(C)=0.10
$$

The policy transformation can be represented as:

| Move | Before MCTS: $P$ | After MCTS: $\pi$ |
|---|---:|---:|
| A | 70% | 15% |
| B | 20% | 75% |
| C | 10% | 10% |

The resulting training example is:

$$
\boxed{
s\longrightarrow \pi=(0.15,0.75,0.10)
}
$$

The Policy Network is trained to predict this improved distribution.

In other words:

> Next time you encounter a similar position, favor B rather than A. You should not need another 1,000 simulations to rediscover this insight.

This is how AlphaZero compresses expensive search results into neural network parameters.

---

## 3. Why Not Train the Policy Network Directly on Q-Values?

There are three important reasons.

### Reason 1: Q-Values Are Not Action Probabilities

A Q-value answers:

> How good is the expected outcome if I choose this action?

A policy answers:

> How should I distribute my action-selection probabilities among the available moves?

These are fundamentally different questions.

Suppose:

$$
Q(A)=0.80,\qquad Q(B)=0.79
$$

The moves have almost identical estimated values.

But how should we convert these values into action probabilities?

We might use Softmax:

$$
P(a|s)=\frac{e^{Q(s,a)/T}}{\sum_b e^{Q(s,b)/T}}
$$

However, we must choose a temperature $T$.

- A low temperature produces a sharp distribution that strongly favors the best-valued move.
- A high temperature produces a more uniform distribution.

**Q-values alone do not specify the appropriate probability distribution.**

MCTS visit counts, on the other hand, already describe how the search **allocated its decisions across actions**.

Q-values describe expected outcomes (result after thinking), while policy probabilities are relative preferences (initial game sense).


### Reason 2: Q-Values Do Not Express How Much Search Produced Them

Consider two actions:

| Move | Q-Value | Visit Count |
|---|---:|---:|
| A | 0.90 | 2 |
| B | 0.80 | 500 |

A has the higher estimated value.

But A has been visited only twice, whereas B has been visited 500 times.

Should we immediately conclude that A is better?

Not necessarily.

A's high Q-value may be based on very limited evidence. B's estimate has been updated through substantially more search.

Of course, more visits do not automatically guarantee that an estimate is correct. MCTS samples are dependent and are not uniformly distributed.

The important distinction is:

**Q represents the current value estimate, while N provides information about how much search has been allocated to that action.**

This helps explain why PUCT considers both estimated value and exploration.

The PUCT selection rule is:

$$
a^*=\arg\max_a
\left[
Q(s,a)+
c_{\text{puct}}P(s,a)
\frac{\sqrt{N(s)}}{1+N(s,a)}
\right]
$$

Here:

- $Q(s,a)$ represents the current estimated value.
- $P(s,a)$ represents the neural network's prior preference.
- $N(s,a)$ represents the number of times the action has been visited.
- $N(s)$ represents the parent's visit count.
- $c_{\text{puct}}$ controls the exploration bonus.

The algorithm does not simply choose the action with the highest Q-value. It balances current evidence against the potential benefit of further exploration.

### Reason 3: Visit Counts Reflect the Behavior of the Entire Search Algorithm

This is the most subtle reason.

During MCTS, actions are repeatedly selected using:

$$
a^*=\arg\max_a(Q+U)
$$

The exploration bonus $U$ depends on both the policy prior and visit statistics.

As a result, the final visit distribution incorporates the effects of:

1. Estimated action values $Q$.
2. Neural network priors $P$.
3. Previous search allocations $N$.
4. The exploration–exploitation tradeoff.
5. The computational budget available to the search.

Therefore, the visit-count distribution is not merely another value estimate.

It represents the **action-selection behavior produced by the search process**.

This is precisely what the Policy Network is intended to learn.

An important qualification: visit counts are not ground-truth action probabilities, nor are they guaranteed to represent the optimal policy. They depend on search budget, priors, and exploration parameters.

Nevertheless, they provide a practical target for learning the improved decision policy generated by MCTS.

---

## 4. The MCTS Policy Target and Temperature

More generally, AlphaZero constructs its search policy using:

$$
\boxed{
\pi(a|s)=
\frac{N(s,a)^{1/\tau}}
{\sum_b N(s,b)^{1/\tau}}
}
$$

where $\tau$ is the temperature parameter.

Its effect is:

| Temperature | Behavior |
|---|---|
| $\tau=1$ | Probabilities are proportional to visit counts |
| $\tau\to0^+$ | Distribution concentrates on the most-visited action |
| Large $\tau$ | Distribution becomes more uniform across visited actions |

The temperature controls how sharply the search visit counts are converted into action probabilities.

This is separate from the exploration parameter $c_{\text{puct}}$, which affects how MCTS performs its search.

The key distinction:

- **PUCT controls which branches receive search effort.**
- **Temperature controls how the resulting visit counts become a policy.**

---

## 5. AlphaZero Actually Trains Two Different Prediction Heads

AlphaZero's neural network has two outputs:

$$
f_\theta(s)=(p,v)
$$

where:

- $p$ is the predicted policy distribution.
- $v$ is the predicted value of the current position.

These two outputs learn from different targets.

| Component | Training Target | Question Being Answered |
|---|---|---|
| Policy Head | MCTS visit distribution $\pi$ | Which actions should I prefer? |
| Value Head | Final game outcome $z$ | How likely is this position to lead to a good result? |

### Policy Head: Learn from Search

The Policy Head is trained to imitate the MCTS policy.

Its cross-entropy loss is:

$$
L_{\text{policy}}
=
-\sum_a\pi(a|s)\log p(a|s)
$$

The goal is to make:

$$
p(a|s)\approx\pi(a|s)
$$

The network learns to produce decisions resembling those of the more computationally expensive search procedure.

### Value Head: Learn from Actual Game Outcomes

The Value Head is trained using the eventual game result.

For a win/draw/loss representation:

$$
z\in\{-1,0,+1\}
$$

The value loss is:

$$
L_{\text{value}}=(z-v)^2
$$

An important distinction:

**The Value Head learns directly from the final self-play outcome $z$, not from an intermediate MCTS Q-value.**

Q-values are estimates accumulated during search, using backed-up value-network evaluations and, when available, actual terminal outcomes.

By contrast, $z$ is the result eventually observed when the self-play game finishes.

### Combined AlphaZero Loss

The overall training objective is:

$$
\boxed{
L(\theta)
=
(z-v)^2
-\sum_a\pi(a|s)\log p(a|s)
+\lambda\|\theta\|^2
}
$$

The three terms represent:

1. **Value Loss:** Predict the final game outcome.
2. **Policy Loss:** Imitate the search-improved policy.
3. **Regularization:** Discourage unnecessarily large model parameters.

This produces two complementary learning signals.

The Policy Head learns **how to act**, while the Value Head learns **how to evaluate positions**.

---

## 6. The Complete AlphaZero Learning Loop

The entire process can be represented without a diagram:

```text
                  Neural Network
                       |
                       v
                 Input state s
                       |
              +--------+--------+
              |                 |
              v                 v
         Policy Head       Value Head
              |                 |
              P                 V
              |                 |
              +--------+--------+
                       |
                       v
                      MCTS
                       |
           Search using P, V, Q, N
                       |
                       v
             Improved policy pi
                       |
                       v
                Self-play game
                       |
                       v
              Final outcome z
                       |
                       v
              Training examples
                 (s, pi, z)
                       |
              +--------+--------+
              |                 |
              v                 v
         Policy learns pi   Value learns z
              |                 |
              +--------+--------+
                       |
                       v
              Updated Network
                       |
                       v
               Repeat the Loop
```

---

## 8. The Deeper Idea: Amortized Search

One of the most important concepts behind AlphaZero is **amortized computation**, specifically *Amortized Search*.

Suppose solving a difficult position requires 1,000 MCTS simulations.

Without learning, an agent might need to repeatedly perform similarly expensive searches whenever it encounters related positions.

But with a neural network, the agent can use those expensive searches as training examples.

Over many positions, the network learns to approximate the decisions that search would have made.

The distinction is:

**Search:**

> Spend computation now to solve this particular decision problem.

**Learning:**

> Use the results of many expensive searches to improve future predictions.

Conceptually:

$$
\boxed{
\text{Expensive Search}
\longrightarrow
\text{Training Data}
\longrightarrow
\text{Learned Parameters}
\longrightarrow
\text{Faster Future Predictions}
}
$$

This does not mean the neural network perfectly memorizes MCTS outcomes or completely eliminates the need for search.

Instead, it learns reusable patterns that make future searches more effective.

The search improves the network, and the improved network helps guide subsequent searches.


### One-Sentence Takeaway

**AlphaZero uses MCTS to turn the neural network's initial intuition into a stronger decision policy, then trains the neural network to imitate that policy—compressing expensive search computation into reusable knowledge.**